In [ ]:
import pandas as pd
import numpy as np 
from scipy.sparse import hstack
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
import faiss
from sentence_transformers import SentenceTransformer
import ast
import os
import pickle
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

Para la recomendación de libros vamos a utilizar FAISS y all-MiniLM-L6-v2 para hacer embedding y que combine descripción y género y que al buscar libros, te haga la búsqueda sobre estas dos columnas.

In [ ]:
# Preparación del texto para embeddings
df['genres'] = df['genres'].apply(lambda g: g if isinstance(g, list) else [])
df['text_for_embedding'] = df['description'] + ". Genres: " + df['genres'].apply(lambda g: ", ".join(g))

# Generación de embeddings
modelo = SentenceTransformer('all-MiniLM-L6-v2')
print("Generating embeddings...")
embeddings = modelo.encode(df['text_for_embedding'].tolist(), convert_to_tensor=False, show_progress_bar=True)

# Crear índice FAISS
dimension = embeddings[0].shape[0]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# Función de recomendación
def recommend_books(user_text, top_n=5):
    vec = modelo.encode([user_text])
        distances, indices = index.search(np.array(vec), top_n)
            results = df.iloc[indices[0]].copy()
                results['score'] = distances[0]
                    return results[['title', 'author', 'genres', 'rating', 'coverImg', 'description', 'score']]

Generating embeddings...


Batches:   0%|          | 0/1791 [00:00<?, ?it/s]/home/codespace/.local/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Batches: 100%|██████████| 1791/1791 [47:12<00:00,  1.58s/it]


In [12]:
# Consulta de prueba
query = "I want to read a science fiction story with philosophical themes"
results = recommend_books(query)

for i, row in results.iterrows():
    print(f"\n\033[1m{row['title']}\033[0m by {row['author']}")
    print(f"Rating: {row['rating']} | Score: {row['score']:.4f}")
    print(f"Genres: {', '.join(row['genres'])}")
    print(f"Description: {row['description'][:300]}...")
    print(f"Cover: {row['coverImg']}")


Nobel Dreams: Power, Deceit, and the Ultimate Experiment by Gary Taubes (Goodreads Author)
Rating: 4.12 | Score: 0.8026
Genres: fiction, nonfiction
Description: A modern-day adventure story demonstrating the chaotic and chancy nature of science that will change perceptions of scientific research. Several colorful personalities are introduced--the Italian physicist Carlo Rubbia chief among them....
Cover: https://i.gr-assets.com/images/S/compressed.photo.goodreads.com/books/1387668146l/3071857.jpg

The Universe Story: From the Primordial Flaring Forth to the Ecozoic Era--A Celebration of the Unfolding of the Cosmos by Brian Swimme, Thomas Berry
Rating: 4.24 | Score: 0.8084
Genres: fiction, nonfiction
Description: From the big bang to the present and into the next millenium, The Universe Story unites science and the humanities in a dramatic exploration of the unfolding of the universe, humanity's evolving place in the cosmos, and the boundless possibilities for our future.
...
Cover: ht

Guardar embeddings y FAISS

In [18]:
# Guardar embeddings
np.save('../data/final_data/embeddings.npy', embeddings)

# Guardar índice FAISS
faiss.write_index(index, '../data/final_data/faiss_index.idx')

Hicimos un cambio en la función de recomendación para que el usuario, al hacer la búsqueda más de una vez con el mismo prompt, no obtenga siempre los mismos resultados. Para no perder calidad, mantenemos el resultado más relevante y variamos aleatoriamente el resto dentro de un grupo pequeño de candidatos muy similares.

In [13]:
# Cargar dataset
df = pd.read_csv('/workspaces/proyecto_libros/data/final_data/df_web.csv')

# Asegurar que los géneros siempre sean listas
df['genres'] = df['genres'].apply(
    lambda g: eval(g) if isinstance(g, str) and g.startswith('[') else (g if isinstance(g, list) else [])
)

# Cargar índice FAISS
index = faiss.read_index('/workspaces/proyecto_libros/data/final_data/faiss_index.idx')

# Cargar modelo
modelo = SentenceTransformer('all-MiniLM-L6-v2')

# Función de recomendación
def recommend_books(user_text, top_n=5, pool_size=15):
    vec = modelo.encode([user_text])
    distances, indices = index.search(np.array(vec), pool_size)
    
    results = df.iloc[indices[0]].copy()
    results['score'] = distances[0]
    
    # Mantener siempre el más relevante
    first_result = results.iloc[[0]]
    
    # Variar los demás
    rest = results.iloc[1:].sample(n=top_n-1, random_state=None)
    
    return pd.concat([first_result, rest]).reset_index(drop=True)

In [14]:
# Consulta de prueba
query = "I want to read a science fiction story with philosophical themes"
results = recommend_books(query)

for i, row in results.iterrows():
    print(f"\n\033[1m{row['title']}\033[0m by {row['author']}")
    print(f"Rating: {row['rating']} | Score: {row['score']:.4f}")
    print(f"Genres: {', '.join(row['genres'])}")
    print(f"Description: {row['description'][:300]}...")
    print(f"Cover: {row['coverImg']}")


Nobel Dreams: Power, Deceit, and the Ultimate Experiment by Gary Taubes (Goodreads Author)
Rating: 4.12 | Score: 0.8026
Genres: fiction, nonfiction
Description: A modern-day adventure story demonstrating the chaotic and chancy nature of science that will change perceptions of scientific research. Several colorful personalities are introduced--the Italian physicist Carlo Rubbia chief among them....
Cover: https://i.gr-assets.com/images/S/compressed.photo.goodreads.com/books/1387668146l/3071857.jpg

Evolution of Insanity by Haresh Daswani (Goodreads Author)
Rating: 4.29 | Score: 0.9260
Genres: fiction
Description: An author having a conversation with his fictional character, or losing control of his character, mind numbing points leading one twists and turns spinning the mind of the reader with hallucinogenic colors, concepts, and eurekas. The short stories begin simplified, and walks together with the author...
Cover: https://i.gr-assets.com/images/S/compressed.photo.goodreads.com/book

/home/codespace/.local/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [15]:
# Consulta de prueba
query = "I want to read a science fiction story with philosophical themes"
results = recommend_books(query)

for i, row in results.iterrows():
    print(f"\n\033[1m{row['title']}\033[0m by {row['author']}")
    print(f"Rating: {row['rating']} | Score: {row['score']:.4f}")
    print(f"Genres: {', '.join(row['genres'])}")
    print(f"Description: {row['description'][:300]}...")
    print(f"Cover: {row['coverImg']}")


Nobel Dreams: Power, Deceit, and the Ultimate Experiment by Gary Taubes (Goodreads Author)
Rating: 4.12 | Score: 0.8026
Genres: fiction, nonfiction
Description: A modern-day adventure story demonstrating the chaotic and chancy nature of science that will change perceptions of scientific research. Several colorful personalities are introduced--the Italian physicist Carlo Rubbia chief among them....
Cover: https://i.gr-assets.com/images/S/compressed.photo.goodreads.com/books/1387668146l/3071857.jpg

The Egg by Andy Weir (Goodreads Author)
Rating: 4.19 | Score: 0.9313
Genres: fantasy, fiction
Description: A short story about the universe....
Cover: https://i.gr-assets.com/images/S/compressed.photo.goodreads.com/books/1431492647l/17563539.jpg

The Universe Story: From the Primordial Flaring Forth to the Ecozoic Era--A Celebration of the Unfolding of the Cosmos by Brian Swimme, Thomas Berry
Rating: 4.24 | Score: 0.8084
Genres: fiction, nonfiction
Description: From the big bang to the prese

/home/codespace/.local/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
